# Aula 09 — Laboratório SQL

**Banco de Dados · Engenharia de Software · 2026/2**

Notebook de preparação para a [Aula 09 — Agregação, agrupamento e subconsultas](https://pfransozi.github.io/bd-2026-02/aula-09.html).

Execute as três células abaixo, nesta ordem. O banco SQLite é criado em memória, com **12 usuários, 11 salas e 20 reservas**, exatamente como na Aula 08. Reexecutar a primeira célula restaura os dados originais.


In [ ]:
import sqlite3

conexao = sqlite3.connect(":memory:")
cursor = conexao.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

cursor.executescript("""
DROP TABLE IF EXISTS reserva;
DROP TABLE IF EXISTS telefone_usuario;
DROP TABLE IF EXISTS sala;
DROP TABLE IF EXISTS usuario;
DROP TABLE IF EXISTS bloco;

CREATE TABLE usuario (
    id_usuario INTEGER PRIMARY KEY,
    nome TEXT NOT NULL
);

CREATE TABLE telefone_usuario (
    id_usuario INTEGER NOT NULL,
    telefone TEXT NOT NULL,
    PRIMARY KEY (id_usuario, telefone),
    FOREIGN KEY (id_usuario)
        REFERENCES usuario(id_usuario)
        ON DELETE CASCADE
);

CREATE TABLE bloco (
    id_bloco INTEGER PRIMARY KEY,
    nome_bloco TEXT NOT NULL UNIQUE
);

CREATE TABLE sala (
    id_sala INTEGER PRIMARY KEY,
    nome TEXT NOT NULL,
    capacidade INTEGER NOT NULL CHECK (capacidade > 0),
    id_bloco INTEGER NOT NULL,
    FOREIGN KEY (id_bloco)
        REFERENCES bloco(id_bloco)
        ON DELETE RESTRICT
);

CREATE TABLE reserva (
    id_reserva INTEGER PRIMARY KEY,
    id_usuario INTEGER NOT NULL,
    id_sala INTEGER NOT NULL,
    data TEXT NOT NULL,
    hora_inicio TEXT NOT NULL,
    situacao TEXT NOT NULL DEFAULT 'ativa'
        CHECK (situacao IN ('ativa', 'cancelada', 'concluida')),
    observacao TEXT,

    UNIQUE (id_sala, data, hora_inicio),
    FOREIGN KEY (id_usuario)
        REFERENCES usuario(id_usuario)
        ON DELETE RESTRICT,
    FOREIGN KEY (id_sala)
        REFERENCES sala(id_sala)
        ON DELETE RESTRICT
);

INSERT INTO bloco (id_bloco, nome_bloco) VALUES
    (1, 'Bloco A'),
    (2, 'Bloco B'),
    (3, 'Bloco C'),
    (4, 'Bloco D');

INSERT INTO usuario (id_usuario, nome) VALUES
    (8, 'Larissa Moura'),
    (17, 'Marina Lopes'),
    (42, 'Gustavo Neri'),
    (55, 'Ana Souza'),
    (71, 'Bruno Ramos'),
    (100, 'João Martins'),
    (108, 'Camila Freitas'),
    (112, 'Rafael Costa'),
    (119, 'Beatriz Lima'),
    (124, 'Diego Martins'),
    (130, 'Pedro Alves'),
    (135, 'Sofia Rocha');

INSERT INTO telefone_usuario (id_usuario, telefone) VALUES
    (8, '41981234567'),
    (17, '41987651223'),
    (17, '41991128080'),
    (42, '41984327710'),
    (55, '41990001122'),
    (71, '41982223344'),
    (100, '41985556677'),
    (108, '41987778899'),
    (112, '41983334455'),
    (119, '41986667788'),
    (124, '41989990011'),
    (135, '41981112233');

INSERT INTO sala (id_sala, nome, capacidade, id_bloco) VALUES
    (118, 'Sala Araucária', 8, 1),
    (119, 'Sala Cedro', 12, 1),
    (120, 'Sala Imbuia', 20, 1),
    (204, 'Sala Ipê', 6, 2),
    (205, 'Sala Pinhão', 4, 2),
    (206, 'Sala Gralha Azul', 14, 2),
    (305, 'Sala Horizonte', 10, 3),
    (306, 'Laboratório Sul', 24, 3),
    (401, 'Laboratório Maker', 16, 4),
    (402, 'Sala Inovação', 30, 4),
    (403, 'Sala Jacarandá', 18, 4);

INSERT INTO reserva
    (id_reserva, id_usuario, id_sala, data, hora_inicio, situacao, observacao)
VALUES
    (1, 17, 204, '2026-10-08', '18:30', 'ativa', 'Reunião de projeto'),
    (2, 42, 118, '2026-10-08', '19:00', 'ativa', NULL),
    (3, 8, 305, '2026-10-09', '18:30', 'concluida', 'Apresentação'),
    (4, 55, 119, '2026-10-09', '20:00', 'cancelada', NULL),
    (5, 17, 118, '2026-10-10', '18:30', 'ativa', 'Estudo em grupo'),
    (6, 71, 204, '2026-10-10', '19:30', 'ativa', NULL),
    (7, 42, 305, '2026-10-11', '18:00', 'concluida', 'Defesa de trabalho'),
    (8, 55, 205, '2026-10-11', '20:00', 'ativa', NULL),
    (9, 100, 120, '2026-10-12', '08:00', 'ativa', 'Encontro da equipe'),
    (10, 108, 206, '2026-10-12', '10:00', 'cancelada', NULL),
    (11, 112, 306, '2026-10-12', '14:00', 'ativa', 'Revisão de sprint'),
    (12, 119, 401, '2026-10-12', '16:00', 'concluida', 'Oficina de prototipação'),
    (13, 124, 402, '2026-10-13', '09:00', 'ativa', NULL),
    (14, 135, 118, '2026-10-13', '18:30', 'ativa', 'Orientação de projeto'),
    (15, 8, 206, '2026-10-13', '19:30', 'cancelada', NULL),
    (16, 17, 305, '2026-10-14', '18:00', 'concluida', 'Apresentação parcial'),
    (17, 42, 401, '2026-10-14', '19:00', 'ativa', NULL),
    (18, 71, 402, '2026-10-14', '20:00', 'ativa', 'Planejamento'),
    (19, 108, 119, '2026-10-15', '18:30', 'concluida', NULL),
    (20, 112, 204, '2026-10-15', '19:30', 'ativa', 'Reunião técnica');
""")
conexao.commit()

for tabela in ("usuario", "sala", "reserva"):
    total = cursor.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    print(f"{tabela}: {total} registros")


## Consultar os dados

A função `consultar(sql)` executa uma consulta SQL no banco e devolve uma tabela. Use-a nos exemplos e exercícios do material.


In [ ]:
import pandas as pd

def consultar(sql):
    return pd.read_sql_query(sql, conexao)


## Teste da conexão

A consulta deve retornar três linhas: 12 reservas ativas, 3 canceladas e 5 concluídas.


In [ ]:
consultar("""
SELECT situacao, COUNT(*) AS total
FROM reserva
GROUP BY situacao
ORDER BY situacao;
""")


## Seus experimentos

Escreva nas próximas células as consultas e exercícios propostos na aula. Para executar um exemplo, use `consultar("""SELECT ...;""")`. Reexecute a célula de inicialização se precisar restaurar os registros originais.


In [ ]:
# Escreva sua consulta aqui:
# consultar("""
# SELECT ...
# """)
